<a href="https://colab.research.google.com/github/TechTinkerKetki/curriculum_bot_iiti/blob/main/02_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import re

# Files are directly in /content
EXTRACTED_DIR = Path("data/processed")

TEXT_PATH = EXTRACTED_DIR / "text_pages.json"
TABLE_PATH = EXTRACTED_DIR / "tables_raw.json"


with open(TEXT_PAGES_PATH, "r") as f:
    text_pages = json.load(f)

with open(TABLES_RAW_PATH, "r") as f:
    tables_raw = json.load(f)

page_text = {p["page"]: p["text"] for p in text_pages}
print("Pages:", len(text_pages))
print("Tables:", len(tables_raw))


Pages: 191
Tables: 184


In [ ]:
def extract_context(text):
    ctx = {}

    if not text:
        return ctx

    if "BTech" in text:
        ctx["program"] = "BTech"

    year = re.search(r"(1st|2nd|3rd|4th)\s+Year", text)
    if year:
        ctx["year"] = year.group(1)

    sem = re.search(r"Semester\s+(I+|IV|V|VI|VII|VIII)", text)
    if sem:
        ctx["semester"] = sem.group(1)

    if "CSE" in text:
        ctx["branch"] = "CSE"

    return ctx


In [ ]:
def table_to_text(table):
    lines = []

    for row in table:
        # keep only non-empty cells
        cells = [str(c).strip() for c in row if str(c).strip()]
        if not cells:
            continue

        line = " ".join(cells)

        # skip obvious junk
        if line.upper().startswith("CONTENTS"):
            continue
        if "Particulars Page No" in line:
            continue

        lines.append(line)

    return "\n".join(lines)


In [ ]:
chunks = []

for t in tables_raw:
    page = t["page"]
    table = t["data"]

    content = table_to_text(table)
    if len(content) < 40:
        continue

    context = extract_context(page_text.get(page, ""))

    chunks.append({
        "content": content,
        "metadata": {
            "page": page,
            **context
        }
    })

print("Chunks created:", len(chunks))


Chunks created: 184


In [ ]:
chunks_out_path = EXTRACTED_DIR / "chunks.json"

with open(chunks_out_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print("Saved chunks to:", chunks_out_path)



Saved chunks.json
